# Interactive sesion with Spark to explore results in Bronze Layer

Running a spark cluster in local mode.

Running Spark in a Python Virtual Environment

In [6]:
import os
import sys

# 1. FORZAR EL AISLAMIENTO ANTES DE CUALQUIER OTRA IMPORTACIÓN
# Removemos cualquier ruta que mencione /opt/spark del sistema para que no interfiera
sys.path = [path for path in sys.path if "opt/spark" not in path]

if ".venv" in sys.executable:
    current_python = sys.executable  
    os.environ["PYSPARK_PYTHON"] = current_python
    os.environ["PYSPARK_DRIVER_PYTHON"] = current_python
    
    # Extraer la raíz del entorno virtual de forma segura
    venv_base = current_python.split("/bin/python")[0]
    
    # Apuntar al PySpark local de Python 3.8
    pyspark_local = f"{venv_base}/lib/python3.8/site-packages"
    os.environ["SPARK_HOME"] = f"{pyspark_local}/pyspark"
    os.environ["PATH"] = f"{os.environ['SPARK_HOME']}/bin:" + os.environ["PATH"]
    
    # Insertar la ruta local al inicio de la lista de búsqueda de Python
    if pyspark_local not in sys.path:
        sys.path.insert(0, pyspark_local)


Build A Spark Session 

In [7]:

# 2. AHORA SÍ IMPORTAMOS (Python buscará exclusivamente en tu .venv)
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
import pandas as pd

# 3. Configurar visualización cómoda de las tablas interactivas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)

# 4. Iniciar sesión local de Spark (4GB de RAM)
builder = SparkSession.builder \
    .appName("Interactive_Bronze_Exploration") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


In [ ]:
# 5. Cargar la tabla Delta de la Capa Bronze
table_path = os.path.expanduser("~/lake/bronze/pid_documents")
df = spark.read.format("delta").load(table_path)

print("¡Sesión interactiva aislada y lista! Ejecuta la siguiente celda.")

¡Sesión interactiva aislada y lista! Ejecuta la siguiente celda.


In [ ]:
# Convertimos las filas a Pandas para activar la tabla interactiva de VS Code
# Excluimos temporalmente las columnas binarias pesadas para mejorar la velocidad
columnas_visibles = [
    "document_number", "drawing_revision", "project_code", 
    "source_filename", "file_size_bytes", "ingested_at", "content_text"
]

df.select(columnas_visibles).limit(10).toPandas()

,document_number,drawing_revision,project_code,source_filename,file_size_bytes,ingested_at,content_text
0,362-09-PR-PID-01470,01,B,362-09-01470_Dexpi.xml,12537819,2026-08-28 18:44:52.468127,"<?xml version=""1.0"" encoding=""utf-8""?>\r\n<PlantModel xmlns:xsi=""http://www.w3.org/2001/XMLSchema-instance"">\r\n <!--Created with INGR ISO15926 P..."
1,362-09-PR-PID-01050,01,B,362-09-01050_Dexpi.xml,11926337,2026-08-28 18:44:52.468127,"<?xml version=""1.0"" encoding=""utf-8""?>\r\n<PlantModel xmlns:xsi=""http://www.w3.org/2001/XMLSchema-instance"">\r\n <!--Created with INGR ISO15926 P..."
2,362-09-PR-PID-01010,01,B,362-09-01010_Dexpi.xml,11819694,2026-08-28 18:44:52.468127,"<?xml version=""1.0"" encoding=""utf-8""?>\r\n<PlantModel xmlns:xsi=""http://www.w3.org/2001/XMLSchema-instance"">\r\n <!--Created with INGR ISO15926 P..."
3,362-92-PR-PID-02265,01,B,362-92-02265_Dexpi.xml,6669430,2026-08-28 18:44:52.468127,"<?xml version=""1.0"" encoding=""utf-8""?>\r\n<PlantModel xmlns:xsi=""http://www.w3.org/2001/XMLSchema-instance"">\r\n <!--Created with INGR ISO15926 P..."


Historial de la tabla delta

In [10]:
try:
    # Instanciar el objeto DeltaTable para acceder a sus funciones de gestión
    delta_table = DeltaTable.forPath(spark, table_path)
    
    # Obtener los últimos eventos del log de transacciones
    history_df = delta_table.history()
    
    # Seleccionar las columnas más importantes para auditoría
    history_df.select("version", "timestamp", "operation", "operationParameters") \
              .show(truncate=False)

    # --- EJEMPLO DE TIME TRAVEL ---
    print("=== LEYENDO UNA VERSIÓN ESPECÍFICA (TIME TRAVEL) ===")

    # Puedes leer la versión 0 (tu primera ingesta exitosa) usando la opción "versionAsOf"
    df_version_0 = spark.read.format("delta").option("versionAsOf", 0).load(table_path)

    print(f"Total de filas en la versión 0: {df_version_0.count()}")
    df_version_0.show(1, vertical=True, truncate=100)

except Exception as e:
    print(f"Error al leer la tabla Delta: {e}")

finally:
    spark.stop()

+-------+-----------------------+---------+-------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                    |
+-------+-----------------------+---------+-------------------------------------------------------+
|0      |2026-08-28 18:45:04.225|WRITE    |{mode -> ErrorIfExists, partitionBy -> ["ingest_date"]}|
+-------+-----------------------+---------+-------------------------------------------------------+

=== LEYENDO UNA VERSIÓN ESPECÍFICA (TIME TRAVEL) ===
Total de filas en la versión 0: 4


-RECORD 0-----------------------------------------------------------------------------------------------------------------------
 bronze_id               | bed7a35f-ff9e-4350-a88b-6e5634240f48                                                                 
 content                 | [3C 3F 78 6D 6C 20 76 65 72 73 69 6F 6E 3D 22 31 2E 30 22 20 65 6E 63 6F 64 69 6E 67 3D 22 75 74 ... 
 content_text            | <?xml version="1.0" encoding="utf-8"?>\r\n<PlantModel xmlns:xsi="http://www.w3.org/2001/XMLSchema... 
 content_hash            | f3857464649b3eff46c0c562fe7e8316518bc2e0d5dca3c447c1c52175edd181                                     
 file_size_bytes         | 12537819                                                                                             
 source_path             | file:/home/dcamacho/dev/bronze_ingestion/data/exports/projectB/362-09-01470_Dexpi.xml                
 source_filename         | 362-09-01470_Dexpi.xml                                                